In [1]:
from vistiq.segment import MicroSAMSegmenter, MicroSAMSegmenterConfig, MicroSAMMerger, MicroSAMMergerConfig
from vistiq.segment import TiledSegmentationFlow, TiledSegmentationFlowConfig, SegmentationFlow, SegmentationFlowConfig
from vistiq.io import ImageLoader, ImageLoaderConfig
from vistiq.core import FuncProcessor, FuncProcessorConfig, Tiler, TilerConfig, Untiler, UntilerConfig
from vistiq.preprocess import ResizeConfig, Resize, RescaleConfig, Rescale, DoG, DoGConfig, PreprocessorConfig, Preprocessor
from vistiq.segment import RegionFilterConfig, RegionFilter, RangeFilterConfig, RangeFilter, RegionAnalyzerConfig, RegionAnalyzer 
from vistiq.utils import ArrayIteratorConfig 
from vistiq.analysis import CoincidenceDetectorConfig, CoincidenceDetector
from vistiq.io import ImageWriterConfig, ImageWriter

from skimage.exposure import rescale_intensity, adjust_sigmoid, adjust_gamma
import stackview
import os
import numpy as np
#from joblib import Parallel, delayed
import math
import logging

2026-05-28 00:55:29,256 - INFO - No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


In [2]:
import vistiq
logger = logging.getLogger(vistiq.__name__)

import torch
logger.info(f"Torch version: {torch.__version__}. Cuda available: {torch.cuda.is_available()}, MPS available: {torch.backends.mps.is_available()}")

2026-05-28 00:55:34,042 - INFO - Torch version: 2.7.1. Cuda available: False, MPS available: True


# Load image

In [3]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
#path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"
path="/Users/khs3z/Documents/SDS_/projects/Siegrist/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif"
path="Animal 1.lif"

scene_index = 0

embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"
embedding_path = "./embeddings"

In [4]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=True,
    #substack="C:1"
)
img, metadata = ImageLoader(ilc).run(path)

2026-05-28 00:55:34,133 - INFO - Loading image from: Animal 1.lif
2026-05-28 00:55:35,058 - INFO - Scenes found: ('Series001', 'Series002', 'Series003')
2026-05-28 00:55:35,161 - INFO - Loaded image: Animal 1.lif scene=0 -> shape=(3, 93, 512, 512) dtype=uint8, channel_names=['Scrib', 'EdU', 'Dpn']
2026-05-28 00:55:35,162 - INFO - Loaded image with shape: (3, 93, 512, 512), dtype: uint8
2026-05-28 00:55:35,164 - INFO - Finished in state Completed()
2026-05-28 00:55:37,283 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-05-28 00:55:39,356 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-05-28 00:55:41,470 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspac

# Preprocess

In [5]:
# Rescale intensity for each channel
scfg = RescaleConfig(
    low=2, 
    high=98, 
    dtype=np.uint8, 
    iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1))
)
simg,_ = Rescale(scfg).run(img, metadata=metadata, verbose=1)

2026-05-28 00:55:35,272 - INFO - Running preprocessor Rescale, on stack of type uint8, True
2026-05-28 00:55:35,358 - INFO - Running Rescale with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-3, -2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None normalize=False dtype=<class 'numpy.uint8'> low=2.0 high=98.0
2026-05-28 00:55:35,358 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-28 00:55:35,358 - INFO - Using Parallel with n_jobs=-1 for 3 iterations
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    0.2s finished
2026-05-28 00:55:35,581 - INFO - Reshaped results to shape (3, 93, 512, 512)
2026-05-28 00:55:35

In [6]:
# Apply gaussian blur, separately for each channel and focal plane
gcfg = FuncProcessorConfig(
    func="skimage.filters.gaussian",
    kwargs={"sigma": 1.0},
    iterator_config=ArrayIteratorConfig(slice_def=(-2,-1))
)

nimg, _ = FuncProcessor(gcfg).run(simg, verbose=1)

2026-05-28 00:55:35,895 - INFO - Running FuncProcessor with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None strict_axis=True dtype=None func=<function gaussian at 0x111b134c0> args=[] kwargs={'sigma': 1.0}
2026-05-28 00:55:35,896 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-28 00:55:35,896 - INFO - Using Parallel with n_jobs=-1 for 279 iterations
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done 140 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 279 out of 279 | elapsed:    0.1s finished
2026-05-28 00:55:36,012 - INFO - Reshaped results to shape (3, 93, 512, 512)
2026-05-2

In [7]:
gamma = 0.2
enimg = np.array([rescale_intensity(adjust_sigmoid(adjust_gamma(i, gamma=gamma)),out_range="uint8") for i in nimg])
print (np.max(enimg))
#emimg = exposure.rescale_intensity(filters.gaussian(exposure.adjust_sigmoid(exposure.adjust_gamma(mimg, gamma=gamma)), sigma=5.0), out_range="uint8")

255


In [8]:
stackview.slice(np.concatenate([simg, rescale_intensity(nimg, out_range="uint8"), enimg], axis=-1))

In [9]:
# Project all channels to one.

pcfg = FuncProcessorConfig(
    func="numpy.max", 
    kwargs={"axis":("C")}, 
    strict_axis=False,
    dtype=np.uint16,
)
c_img, c_metadata = FuncProcessor(pcfg).run(enimg, metadata=metadata)
metadata, c_metadata

2026-05-28 00:55:37,623 - INFO - Running FuncProcessor with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None strict_axis=False dtype=<class 'numpy.uint16'> func=<function max at 0x11112ec70> args=[] kwargs={'axis': 'C'}
2026-05-28 00:55:37,623 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-28 00:55:37,623 - INFO - Mapped axis letters C to axis indices (0,)
2026-05-28 00:55:37,629 - INFO - Converted results to uint16
2026-05-28 00:55:37,629 - INFO - Dropping axes. New axes: ['Z', 'Y', 'X']
2026-05-28 00:55:37,629 - INFO - Updating metadata with new shape ratio: [1. 1. 1.]
2026-05-28 00:55:37,630 - INFO - Metadata updated in FuncProcessor: 4 key(s) chan

({'scene_index': 0,
  'dim_order': 'CZYX',
  'axes': ['C', 'Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (3, 93, 512, 512),
  'dims': <Dimensions [C: 3, Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)},
 {'scene_index': 0,
  'dim_order': 'ZYX',
  'axes': ['Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (93, 512, 512),
  'dims': <Dimensions [Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)})

In [10]:
stackview.slice(c_img, continuous_update=True)

# Segment

In [11]:
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)
microsam = MicroSAMSegmenter(mscfg)


rfcfg = RegionFilterConfig(
    filters=[
        RangeFilter(
            RangeFilterConfig(
                attribute="cross_sectional_area", 
                range=(3000, np.inf)
            )
        ),
        RangeFilter(
            RangeFilterConfig(
                attribute="aspect_ratio", 
                range=(0.5, 1.0)
            )
        ),
    ]
)

rf = RegionFilter(rfcfg)

tsfcfg = TiledSegmentationFlowConfig(
    segmenter = microsam,
    region_filter = rf,
    tile_factor=(6,6),
    resize_factor=(0.25,0.25)
)
labels = TiledSegmentationFlow(tsfcfg).run(c_img, metadata=c_metadata, workers=4, config=tsfcfg)

Using apple MPS device.


2026-05-28 00:55:38,631 - INFO - RegionAnalyzer not provided, using default RegionAnalyzer with properties: ['label', 'centroid', 'cross_sectional_area', 'aspect_ratio']
2026-05-28 00:55:38,632 - INFO - Segmenter config: classname='Configurable' package='vistiq.core' version=None command_group=None segmenter=MicroSAMSegmenter(classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None model_type='vit_l_lm' checkpoint=None embedding_path='./embeddings' pred_iou_thresh=(0.88,) stability_score_thresh=(0.95,) box_nms_thresh=(0.7,) crop_nms_thresh=(0.7,) min_mask_region_area=(0,) output_mode=('instance_segmentation',) with_background=(True,) device=None) merger=None region_analyzer=RegionAnalyzer(classname

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-05-28 00:56:32,827 - INFO - Running LabelRemover with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=False split_axis=None split_channels=False rename_channel=None remap=True
2026-05-28 00:56:32,827 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-28 00:56:33,399 - INFO - Finished in state Completed()
2026-05-28 00:56:33,612 - INFO - Finished in state Completed()
2026-05-28 00:56:33,992 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a17cac-ce5d-724f-8000-7cb231cbd4d9/set_state "HTTP/1.1 201 Created"
2026-05-28 00:56:33,997 - INFO - Finished in state Completed()
2026-05-28 00:56:34,579 - INFO 

In [12]:
l1=labels==13107
l2=labels>52767
np.unique(labels), labels.shape, l1.shape, l2.shape

(array([    0, 13107, 26214, 39321, 52428, 65535], dtype=uint16),
 (93, 512, 512),
 (93, 512, 512),
 (93, 512, 512))

In [13]:
stackview.blend(img[0].astype("uint16"), labels.astype("uint64"), blend_factor=40)

In [14]:
stackview.orthogonal(labels.astype("uint64"))

# Analyze regions

In [15]:
racfg = RegionAnalyzerConfig(
    properties=["volume", "cross_sectional_area", "bbox", "aspect_ratio"],
    iterator_config = ArrayIteratorConfig(slice_def=()),
    output_type="dataframe"
)
ra = RegionAnalyzer(racfg)

measurements = ra.run(labels, metadata=c_metadata)

2026-05-28 00:56:50,831 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'cross_sectional_area', 'bbox', 'aspect_ratio', 'area']
2026-05-28 00:56:50,832 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-28 00:56:50,850 - INFO - RegionAnalyzer: Applying scale: (-0.9999284782608696, 0.3000933463796477, 0.3000933463796477)
2026-05-28 00:56:51,194 - INFO - Identified 5 regions, return as dataframe
2026-05-28 00:56:51,195 - INFO - Finished in state Completed()
2026-05-28 00:56:51,197 - INFO - Finished in state Completed()


In [16]:
measurements

,bbox-0,bbox-1,bbox-2,bbox-3,bbox-4,bbox-5,area,aspect_ratio,cross_sectional_area
label,,,,,,,,,
13107,0,0,36,93,512,506,-126652.746960,0.581803,9897.516442
26214,10,0,38,93,512,476,-350774.344081,0.757549,5848.237714
39321,16,232,122,93,512,475,-11449.713485,0.688303,878.586497
52428,17,233,126,93,512,474,-342308.693431,0.699825,5456.674154
65535,21,249,161,81,316,280,-42.413350,0.048694,9.005602


# Save label

In [17]:
c_metadata["channel_names"] = ["Brain Lobes"]

In [21]:
imc = ImageWriterConfig()
outpath = ".".join(path.split(".")[:-1]) + f"-scene-{scene_index}.tif"
ImageWriter(imc).run(labels, outpath, metadata=c_metadata)

2026-05-28 01:01:28,005 - INFO - ImageWriter: using config: classname='Configurable' package='vistiq.core' version=None command_group=None path='.' format='tif' overwrite=False writer=None split_channels=False extension='tif'
2026-05-28 01:01:28,005 - INFO - Preparing to save image with metadata: {'scene_index': 0, 'dim_order': 'ZYX', 'axes': ['Z', 'Y', 'X'], 'channel_names': ['Brain Lobes'], 'channel_axis': 0, 'shape': (93, 512, 512), 'dims': <Dimensions [Z: 93, Y: 512, X: 512]>, 'pixel_unit': 'um', 'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477), 'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)}
2026-05-28 01:01:28,029 - INFO - Saved image to [PosixPath('Animal 1-scene-0.Brain Lobes.tif')]
2026-05-28 01:01:28,030 - INFO - Finished in state Completed()
